# 02 - Baseline Model: Random Forest
Predicting ECD fragment ion intensity using a Random Forest regressor.
Target variable is base-peak–normalized intensity (each spectrum scaled to its most intense peak, so values lie in [0, 1]).

In [ ]:
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split

## 1. Load & Parse Data
Rebuild the long-format `ion_df` from the raw parquet (same logic as EDA notebook).

In [ ]:
df = pd.read_parquet("data_for_student.parquet").merge(
    pd.read_parquet("metadata_for_student.parquet"), on=['raw_file', 'scan'], how='left'
)
print(f"Loaded {len(df):,} spectra and merged")

In [ ]:

# get enzyme from raw_file
enzyme_map = {
    "LysC": "LysC",
    "chymo": "chymo",
    "LysN": "LysN",
    "tryps": "tryps",
    "GluC": "GluC",
}


def extract_enzyme(raw_file):
    for name in enzyme_map:
        if name in raw_file:
            return name
    return "Unknown"


df["enzyme"] = df["raw_file"].apply(extract_enzyme)
print(df["enzyme"].value_counts())

In [ ]:
# Base-Peak Normalization
df["intensity"] = df["intensity"].apply(
    lambda x: [i / max(x) for i in x] if len(x) > 0 and max(x) > 0 else [0.0] * len(x)
)

In [ ]:
# Downsample by whole peptides (set FRAC = 1.0 for the full dataset).
# Done here, before expansion, so the negative space stays cheap to build and
# whole peptides stay together (no leakage between train/val/test).
FRAC = 0.2
sampled = df["modified_sequence"].drop_duplicates().sample(frac=FRAC, random_state=42)
df = df[df["modified_sequence"].isin(set(sampled))].reset_index(drop=True)
df["clean_sequence"] = df["modified_sequence"].str.replace(r"\[.*?\]", "", regex=True)

# Explode to long format INCLUDING the negative space.
# The raw data only stores *matched* (observed) ions, all with positive intensity.
# For every spectrum we enumerate the full grid of theoretically possible ECD
# fragments — {C, z, Z} x cleavage sites 1..L-1 x the charge states seen in that
# spectrum — and assign intensity 0 to the ones that were NOT observed. Without this
# the model never learns to keep absent fragments low, and the per-spectrum SA/PCC
# can never penalise a false-positive peak, so both metrics are optimistically biased.
ION_TYPES = [("C", 1), ("z", 2), ("Z", 3)]


def explode_with_negatives(df):
    cols = [
        "spectrum_id", "sequence", "enzyme", "ion_type", "ion_type_encoded",
        "fragment_number", "ion_charge", "pep_length", "relative_position", "intensity",
    ]
    rows = []
    for sid, seq, clean, ions, intens, enz in zip(
        df.index, df["modified_sequence"], df["clean_sequence"],
        df["matched_ions"], df["intensity"], df["enzyme"],
    ):
        if len(ions) == 0:
            continue
        observed = dict(zip(ions, intens))
        charges = sorted({int(i.split("^")[1]) if "^" in i else 1 for i in ions})
        L = len(clean)
        for ion_type, type_enc in ION_TYPES:
            for fn in range(1, L):  # cleavage sites 1..L-1
                for charge in charges:
                    ion = f"{ion_type}{fn}" if charge == 1 else f"{ion_type}{fn}^{charge}"
                    rows.append((
                        sid, seq, enz, ion_type, type_enc, fn, charge,
                        L, fn / L, observed.get(ion, 0.0),
                    ))
    return pd.DataFrame(rows, columns=cols)


ion_df = explode_with_negatives(df)
n_obs = int((ion_df["intensity"] > 0).sum())
print(
    f"Total fragment slots: {len(ion_df):,} "
    f"(observed: {n_obs:,} | negative space: {len(ion_df) - n_obs:,})"
)
ion_df.head(3)

## 2. Feature Engineering: Local window
CHANGE: swapped global amino acid count for local window (= 4 residues around cleave site, more relevant for intensity)

In [ ]:
AA_INDEX = {aa: idx for idx, aa in enumerate("ACDEFGHIKLMNPQRSTVWY")}

clean_seq = ion_df["sequence"].str.replace(r"\[.*?\]", "", regex=True)
i = ion_df["fragment_number"].values
n = clean_seq.str.len().values
clean_arr = clean_seq.values


def get_residue_vec(clean_arr, pos_arr, length_arr):
    result = np.full(len(clean_arr), -1, dtype=int)
    for j, (seq, p, l) in enumerate(zip(clean_arr, pos_arr, length_arr)):
        if 0 <= p < l:
            result[j] = AA_INDEX.get(seq[p], -1)
    return result


ion_df["res_m2"] = get_residue_vec(clean_arr, i - 2, n)
ion_df["res_m1"] = get_residue_vec(clean_arr, i - 1, n)
ion_df["res_0"] = get_residue_vec(clean_arr, i, n)
ion_df["res_p1"] = get_residue_vec(clean_arr, i + 1, n)

## 3. Prepare Features and Target

In [ ]:
FEATURE_COLS = [
    "ion_type_encoded",
    "fragment_number",
    "ion_charge",
    "pep_length",
    "relative_position",
    "res_m2",
    "res_m1",
    "res_0",
    "res_p1",
]

X = ion_df[FEATURE_COLS].values
y = ion_df["intensity"].values

print(f"Feature matrix shape: {X.shape}")
print(f"Target range (normalized): {y.min():.2f} – {y.max():.2f}")

## 3.5 Note
Peptide downsampling (`FRAC`) and the negative-space expansion now happen together in the load/parse cell above, so whole peptides are kept intact before the train/val/test split.

## 4. Train / Test Split
70-15-15, `.stratify` accounts for enzymes.

In [ ]:
y = ion_df["intensity"].values

seq_enzyme = ion_df.drop_duplicates("sequence")[["sequence", "enzyme"]]

train_seqs, temp_seqs = train_test_split(
    seq_enzyme["sequence"],
    test_size=0.30,
    random_state=42,
    stratify=seq_enzyme["enzyme"],
)
val_enzyme = seq_enzyme[seq_enzyme["sequence"].isin(temp_seqs)].drop_duplicates(
    "sequence"
)
val_seqs, test_seqs = train_test_split(
    temp_seqs, test_size=0.50, random_state=42, stratify=val_enzyme["enzyme"]
)
train_seqs, val_seqs, test_seqs = set(train_seqs), set(val_seqs), set(test_seqs)

train_mask = ion_df["sequence"].isin(train_seqs)
val_mask = ion_df["sequence"].isin(val_seqs)
test_mask = ion_df["sequence"].isin(test_seqs)

X_train = ion_df.loc[train_mask, FEATURE_COLS].values
X_val = ion_df.loc[val_mask, FEATURE_COLS].values
X_test = ion_df.loc[test_mask, FEATURE_COLS].values
y_train = y[train_mask.to_numpy()]
y_val = y[val_mask.to_numpy()]
y_test = y[test_mask.to_numpy()]

print(f"Train: {X_train.shape[0]:,} samples ({len(train_seqs):,} peptides)")
print(f"Val:   {X_val.shape[0]:,} samples ({len(val_seqs):,} peptides)")
print(f"Test:  {X_test.shape[0]:,} samples ({len(test_seqs):,} peptides)")

## 5. Train Random Forest
`n_jobs=-1` uses all CPU cores. `n_estimators=100` is a solid default for a baseline.
Training on ~7M samples will take a few minutes.

In [ ]:
# I LOVE SK LEARN -> we have such simple model swap opportunity omg
MODEL = "xgboost"  # "xgboost" or "randomforest" or "random" or "mean"

if MODEL == "xgboost":
    from xgboost import XGBRegressor

    rf = XGBRegressor(
        n_estimators=300,
        max_depth=7,
        learning_rate=0.05,
        n_jobs=-1,
        random_state=42,
        verbosity=1,
        early_stopping_rounds=15,
    )
    rf.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=True)
elif MODEL == "randomforest":
    from sklearn.ensemble import RandomForestRegressor

    rf = RandomForestRegressor(
        n_estimators=100,
        max_depth=12,
        min_samples_leaf=10,
        n_jobs=-1,
        random_state=42,
        verbose=1,
    )
    rf.fit(X_train, y_train)
elif MODEL == "mean":
    from sklearn.dummy import DummyRegressor
    rf = DummyRegressor(strategy="mean")
    rf.fit(X_train, y_train)
elif MODEL == "random":
    from sklearn.base import BaseEstimator, RegressorMixin

    class RandomUniformRegressor(BaseEstimator, RegressorMixin):
        def fit(self, X, y):return self
        def predict(self, X):return np.random.uniform(0, 1, size=len(X))
    rf = RandomUniformRegressor()
    rf.fit(X_train, y_train)
else: print("Error")


print("Training complete.")

## 6. Evaluate

In [ ]:
from sklearn.metrics import mean_absolute_error, r2_score


def spectral_angle(true_vals, pred_vals):
    # Normvector
    bottom_product = np.linalg.norm(true_vals) * np.linalg.norm(pred_vals)
    if bottom_product < 1e-9:
        return 0.0
    cosine_val = np.clip(np.dot(true_vals, pred_vals) / bottom_product, -1.0, 1.0)
    return 1.0 - (2.0 / np.pi) * np.arccos(cosine_val)


def calculate_pearson(true_vals, pred_vals):
    # Guard against zero vectors
    if np.std(true_vals) < 1e-9 or np.std(pred_vals) < 1e-9:
        return np.nan
    return np.corrcoef(true_vals, pred_vals)[0, 1]


#######################Model evaluation
y_pred = rf.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print(f"MAE (Normalized): {mae:.4f}")
print(f"R²  (Normalized): {r2:.4f}")

###get raw data for Spectral anlge
results_df = ion_df.loc[test_mask].copy()
results_df["actual_intensity"] = y_test
results_df["predicted_intensity"] = y_pred

spectral_angles = []
pearson_correlations = []

for spec_id, peptide_data in results_df.groupby("spectrum_id"):
    actual_list = peptide_data["actual_intensity"].values
    predicted_list = peptide_data["predicted_intensity"].values

    spectral_angles.append(spectral_angle(actual_list, predicted_list))
    pearson_correlations.append(calculate_pearson(actual_list, predicted_list))

sa_arr = np.array(spectral_angles)
pc_arr = np.array(pearson_correlations)

print(f"Median Spectral Angle:      {np.median(sa_arr):.4f}")
print(f"Median Pearson Correlation: {np.nanmedian(pc_arr):.4f}")

## 7. Predicted vs Actual Plot

In [ ]:
# Subsample for plotting (plotting 1.8M points is slow)
rng = np.random.default_rng(42)
idx = rng.choice(len(y_test), size=10_000, replace=False)

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(y_test[idx], y_pred[idx], alpha=0.2, s=5, color="steelblue")

lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
ax.plot(lims, lims, "r--", linewidth=1, label="Perfect prediction")

ax.set_xlabel("Actual Normalized Intensity")
ax.set_ylabel("Predicted Normalized Intensity")
ax.set_title(f"Random Forest — Predicted vs Actual\nR² = {r2:.3f}, MAE = {mae:.3f}")
ax.legend()
plt.tight_layout()
plt.savefig("predicted_vs_actual.png", dpi=150)
plt.show()

## 8. Feature Importance

In [ ]:
importances = rf.feature_importances_
feat_names = FEATURE_COLS

sorted_idx = np.argsort(importances)[::-1]
top_n = 9

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(range(top_n), importances[sorted_idx[:top_n]], color="steelblue")
ax.set_xticks(range(top_n))
ax.set_xticklabels([feat_names[i] for i in sorted_idx[:top_n]], rotation=45, ha="right")
ax.set_ylabel("Importance")
ax.set_title("Top 15 Feature Importances (Random Forest)")
plt.tight_layout()
plt.savefig("feature_importance.png", dpi=150)
plt.show()

## 10. Summary

See cell output above for metrics.

**Features used:** ion type, fragment number, charge, peptide length, relative position, local cleavage site window.
